# Supervise iterative training

Keep a model and optimizer in per-trial artifact state, inspect a pause boundary, and understand a graph-scoped supervisor's selection policy. You should know a PyTorch training loop and the artifact/resource distinction from the inference lesson. Mutable training models belong to trials; the dataset is shared read-only by convention.

## Before you begin

Run the cells in order in a fresh kernel. Keep the complete repository and bundled tutorial data together. The setup below locates your checkout; in Colab it accepts a complete source ZIP. The notebook imports the adjacent Python implementation, so operators and resources have one canonical source.


In [ ]:
from pathlib import Path
import importlib.util
import io
import subprocess
import sys
import zipfile

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / "jayrun").is_dir() and (p / "tutorials").is_dir()), None)
if ROOT is None and IN_COLAB:
    from google.colab import files
    uploaded = files.upload()  # Upload the complete Jayrun project ZIP, not just this notebook.
    if len(uploaded) != 1:
        raise ValueError("upload exactly one project ZIP")
    destination = Path("/content/jayrun_tutorial_source")
    destination.mkdir(exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as archive:
        if sum(item.file_size for item in archive.infolist()) > 500_000_000:
            raise ValueError("archive exceeds this notebook's extraction limit")
        for item in archive.infolist():
            path = (destination / item.filename).resolve()
            if not path.is_relative_to(destination.resolve()) or (item.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError("unsafe archive path or symbolic link")
        archive.extractall(destination)
    candidates = [destination, *destination.iterdir()]
    ROOT = next((p for p in candidates if (p / "jayrun").is_dir() and (p / "tutorials").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Run Jupyter from the extracted project, or upload the full project ZIP in Colab.")
sys.path.insert(0, str(ROOT))
print("Tutorial source:", ROOT)


## Prepare the environment

The next cell checks the packages this lesson uses. Locally, install missing tutorial packages before continuing. In Colab, it installs only missing non-PyTorch packages. For PyTorch lessons, provide a suitable existing PyTorch build; the cell does not replace it.


In [ ]:
required = {'packaging': 'packaging>=25', 'numpy': 'numpy', 'torch': 'torch'}
missing = [requirement for name, requirement in required.items() if importlib.util.find_spec(name) is None]
# Colab provides PyTorch. Never replace an existing CPU/CUDA installation here.
if "torch" in missing:
    raise RuntimeError("Install the PyTorch build appropriate for your hardware before running this lesson.")
if missing:
    if not IN_COLAB:
        raise RuntimeError(f"Install these optional tutorial dependencies first: {missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
from packaging.version import Version
import packaging
if Version(packaging.__version__) < Version("25"):
    raise RuntimeError("This source archive requires packaging >= 25; update it explicitly before proceeding.")
import torch
torch.set_num_threads(1)  # Keep the small CPU demonstration bounded.


## 1. Inspect one training iteration

TrainEpoch consumes dataset and TrainingState, then outputs the updated state under the same artifact identity. One graph iteration performs one epoch. The optimizer is created once and retained inside TrainingState so momentum survives later iterations. A pause requests a lifecycle boundary; it does not release the model or its placement.


In [ ]:
import asyncio
import inspect
from jayrun import ArtifactContext, ConfigContext, Engine
from jayrun.context import ContextState
from jayrun.settings import ContextSettings
from tutorials import mnist_training as lesson
from tutorials.mnist_data import load_data

print(inspect.getsource(lesson.TrainEpoch))
print(inspect.getsource(lesson.build_training_graph))


## 2. Pause one run, inspect it, then Stop

Submit a small run with no fixed iteration limit and a pause at epoch two. After PAUSED, the latest record should show epoch=2. Stop prevents another graph iteration and drains accepted work, producing FINISHED with stop_requested. Stop already handles a paused run; a separate resume is unnecessary here.


In [ ]:
def inspect_pause_boundary():
    data = load_data(training_samples=64, validation_samples=32)
    graph, dataset, state, train = lesson.build_training_graph()
    with Engine() as engine:
        run = engine.submit(
            graph, ArtifactContext({dataset: data, state: lesson.new_state(11)}),
            ConfigContext({train.pause_at: (2,)}),
            settings=ContextSettings(max_iterations=None, record_history_limit=4),
        )
        try:
            run.wait(ContextState.PAUSED, timeout=30)
            record = run.records("training")[-1].value
            assert record["epoch"] == 2
            print("At review:", dict(record))
            run.stop()
            run.wait(timeout=30)
            assert run.state is ContextState.FINISHED and run.report.data.stop_requested
            return {"state": run.state.value, "epoch": run.artifact(state).value.epoch,
                    "stop_requested": run.report.data.stop_requested}
        finally:
            if not run.state.is_terminal:
                run.abort()

boundary = await asyncio.to_thread(inspect_pause_boundary)
print(boundary)
assert boundary == {"state": "finished", "epoch": 2, "stop_requested": True}


## 3. Understand the selection sequence

The complete example starts two trials and compares them at epoch two. It holds the best, aborts the other, waits for finalization, then submits a replacement. A second comparison promotes the best to epoch four; the last selection requests Stop.

SelectTrials runs with Supervisor(training_graph), so it can inspect and control that graph's runs. The surrounding application submits the replacement: a Supervisor does not gain Controller submission authority. Read the selection code with that division of responsibility in mind.


In [ ]:
print(inspect.getsource(lesson.SelectTrials))


In [ ]:
REAL_MNIST = False
DEVICE = "cpu"
result = await asyncio.to_thread(lesson.run_demo, real_mnist=REAL_MNIST, device=DEVICE)
print(result)
assert result["trials"] == 3
assert sorted(result["outcomes"]) == ["aborted", "aborted", "finished"]
assert result["winner_epoch"] == 4 and result["stop_requested"]


## Try it yourself

Change the small pause exercise to pause at epoch three and update its expected epoch. Inspect how the optimizer update count changes. To continue instead of finishing at the boundary, call resume and provide a later pause or finite iteration limit so the run still terminates.

The full selection keeps at most two live models per cohort. Wait for aborted trials before replacing them because paused runs retain capacity. The reservation is not measured VRAM enforcement. Equal epoch budgets make the comparison meaningful within this example, but fixture accuracy is not real-MNIST model quality.
